In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
!pip install -q --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.0/626.0 kB 13.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.16.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [5]:
# Cell 1: Load extension after upgrading
%load_ext autoreload
%autoreload 2

In [6]:
import os
import sys

# Replace with your actual GitHub username and repository name
REPO_URL = "https://github.com/amitkhedar30/adversarial-attack-defense.git"
REPO_NAME = "adversarial-attack-defense"

# Clone if not already present in the Kaggle working directory
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

# Add repository root to Python path so custom imports work
if f"/kaggle/working/{REPO_NAME}" not in sys.path:
    sys.path.append(f"/kaggle/working/{REPO_NAME}")

print("Repository is ready!")

Repository is ready!


In [7]:
# Pull the latest code from GitHub
!cd {REPO_NAME} && git pull

# Install project dependencies
!pip install -q -r {REPO_NAME}/requirements.txt

remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 118 (delta 30), reused 4 (delta 4), pack-reused 68 (from 2)
Receiving objects: 100% (118/118), 159.79 MiB | 38.63 MiB/s, done.
Resolving deltas: 100% (49/49), done.
From https://github.com/amitkhedar30/adversarial-attack-defense
   4fcb06a..d15266d  main       -> origin/main
Updating 4fcb06a..d15266d
Updating files: 100% (22/22), done.
Fast-forward
 .gitattributes                                  |   0
 README.md                                       | 120 ++++++++
 dashboard/README_DASHBOARD.md                   |  87 ++++++
 dashboard/app.py                                | 394 ++++++++++++++++++++++++
 dashboard/assets/acr_summary.csv                |   2 +
 dashboard/assets/certification_radii.csv        | 251 +++++++++++++++
 dashboard/assets/certified_accuracy_curve.png   | Bin 0 -> 158914 bytes
 dashboard/assets/cw_gallery.pt       

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Import your team's custom modules from the GitHub repository
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from utils.checkpoint import save_checkpoint, load_latest_checkpoint

# 1. Configuration & Hyperparameters
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Running on device: {DEVICE}")

# 2. Initialize Data, Model, and Optimizer
train_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model = get_cifar10_resnet18(num_classes=10).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 3. Handle Checkpointing (Resumes automatically if Kaggle crashes)
start_epoch = load_latest_checkpoint(model, optimizer, "Standard_ResNet18")

# 4. The Standard Training Loop
print("[*] Starting standard baseline training...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    scheduler.step()
    
    # Calculate Epoch Metrics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")
    
    # Save checkpoint at the end of each epoch
    save_checkpoint(model, optimizer, epoch, "Standard_ResNet18")

# 5. Save Final Export Weights
torch.save(model.state_dict(), "/kaggle/working/resnet18_standard_cifar10.pt")
print("[*] Baseline training complete! Final model saved.")

In [ ]:
import torch
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from attacks.whitebox import fgsm_attack, pgd_attack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

# 1. Load the Test Data and Model
_, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model = get_cifar10_resnet18(num_classes=10)

# Load the weights you just trained!
model.load_state_dict(torch.load("/kaggle/working/resnet18_standard_cifar10.pt"))
model.to(DEVICE)
model.eval() # Crucial: Set model to evaluation mode

# 2. Evaluation Metrics
correct_clean = 0
correct_fgsm = 0
correct_pgd = 0
total = 0

print("[*] Evaluating baseline model against attacks...")
print("This may take a minute or two as it generates attacks...")

for inputs, labels in test_loader:
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    total += labels.size(0)
    
    # A. Clean Accuracy (No Attack)
    with torch.no_grad():
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        correct_clean += predicted.eq(labels).sum().item()
        
    # B. FGSM Attack (epsilon = 8/255 is the standard benchmark)
    inputs_fgsm = fgsm_attack(model, inputs, labels, epsilon=8/255)
    with torch.no_grad():
        outputs_fgsm = model(inputs_fgsm)
        _, pred_fgsm = outputs_fgsm.max(1)
        correct_fgsm += pred_fgsm.eq(labels).sum().item()
        
    # C. PGD Attack (iters=10 is the standard quick-test)
    inputs_pgd = pgd_attack(model, inputs, labels, epsilon=8/255, alpha=2/255, iters=10)
    with torch.no_grad():
        outputs_pgd = model(inputs_pgd)
        _, pred_pgd = outputs_pgd.max(1)
        correct_pgd += pred_pgd.eq(labels).sum().item()

# 3. Print the Devastation
print("-" * 30)
print(f"Clean Test Accuracy: {100. * correct_clean / total:.2f}%")
print(f"FGSM Accuracy:       {100. * correct_fgsm / total:.2f}%")
print(f"PGD Accuracy:        {100. * correct_pgd / total:.2f}%")
print("-" * 30)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from utils.checkpoint import save_checkpoint, load_latest_checkpoint
from attacks.whitebox import pgd_attack

# 1. Configuration & Setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 50

print(f"[*] Starting PGD Adversarial Training on device: {DEVICE}")

# 2. Data Loaders & Model Initialization
train_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model_robust = get_cifar10_resnet18(num_classes=10).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_robust.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 3. Resume Checkpoint Logic (Fixed function signature)
start_epoch = load_latest_checkpoint(model_robust, optimizer, "Robust_ResNet18")

# 4. PGD Adversarial Training Loop
for epoch in range(start_epoch, EPOCHS):
    model_robust.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # DYNAMIC ATTACK: Generate PGD perturbations on the training batch
        adv_inputs = pgd_attack(
            model_robust, 
            inputs, 
            labels, 
            epsilon=8/255, 
            alpha=2/255, 
            iters=10
        )
        
        # Forward & Backward pass on ADVERSARIAL inputs
        optimizer.zero_grad()
        outputs = model_robust(adv_inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    scheduler.step()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Robust Loss: {epoch_loss:.4f} | Robust Acc: {epoch_acc:.2f}%")
    
    # Save checkpoint (Fixed function signature)
    save_checkpoint(model_robust, optimizer, epoch, "Robust_ResNet18")

# 5. Export Final Robust Model Weights
torch.save(model_robust.state_dict(), "/kaggle/working/resnet18_robust_pgd_cifar10.pt")
print("[*] PGD Adversarial Training complete! Final robust model saved.")

In [ ]:
import torch
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from attacks.whitebox import fgsm_attack, pgd_attack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

# 1. Load Data & Robust Model
_, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model_robust = get_cifar10_resnet18(num_classes=10)
model_robust.load_state_dict(torch.load("/kaggle/working/resnet18_robust_pgd_cifar10.pt"))
model_robust.to(DEVICE)
model_robust.eval()

correct_clean = 0
correct_fgsm = 0
correct_pgd = 0
total = 0

print("[*] Evaluating ROBUST model (Model B) against attacks...")

for inputs, labels in test_loader:
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    total += labels.size(0)
    
    # Clean Accuracy
    with torch.no_grad():
        outputs = model_robust(inputs)
        _, predicted = outputs.max(1)
        correct_clean += predicted.eq(labels).sum().item()
        
    # FGSM Attack
    inputs_fgsm = fgsm_attack(model_robust, inputs, labels, epsilon=8/255)
    with torch.no_grad():
        outputs_fgsm = model_robust(inputs_fgsm)
        _, pred_fgsm = outputs_fgsm.max(1)
        correct_fgsm += pred_fgsm.eq(labels).sum().item()
        
    # PGD Attack
    inputs_pgd = pgd_attack(model_robust, inputs, labels, epsilon=8/255, alpha=2/255, iters=10)
    with torch.no_grad():
        outputs_pgd = model_robust(inputs_pgd)
        _, pred_pgd = outputs_pgd.max(1)
        correct_pgd += pred_pgd.eq(labels).sum().item()

print("=" * 40)
print(f"Robust Model Clean Accuracy: {100. * correct_clean / total:.2f}%")
print(f"Robust Model FGSM Accuracy:  {100. * correct_fgsm / total:.2f}%")
print(f"Robust Model PGD Accuracy:   {100. * correct_pgd / total:.2f}%")
print("=" * 40)

In [ ]:
# ============================================================
# TRADES Adversarial Training
# Zhang et al. 2019, "Theoretically Principled Trade-off between
# Robustness and Accuracy" (ICML)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from utils.checkpoint import save_checkpoint, load_latest_checkpoint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 50
EPSILON = 8 / 255
STEP_SIZE = 2 / 255
PERTURB_STEPS = 10
BETA = 6.0   # weight on the robustness (KL) term

print(f"[*] Starting TRADES Adversarial Training on device: {DEVICE}")

train_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model_trades = get_cifar10_resnet18(num_classes=10).to(DEVICE)
optimizer = optim.SGD(model_trades.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = load_latest_checkpoint(model_trades, optimizer, "TRADES_ResNet18")


def trades_loss(model, x_natural, y, optimizer, step_size=STEP_SIZE,
                 epsilon=EPSILON, perturb_steps=PERTURB_STEPS, beta=BETA):
    model.eval()

    x_adv = x_natural.detach() + 0.001 * torch.randn_like(x_natural)
    x_adv = torch.clamp(x_adv, 0.0, 1.0)

    for _ in range(perturb_steps):
        x_adv.requires_grad_()
        with torch.enable_grad():
            loss_kl = F.kl_div(
                F.log_softmax(model(x_adv), dim=1),
                F.softmax(model(x_natural), dim=1),
                reduction="sum",
            )
        grad = torch.autograd.grad(loss_kl, [x_adv])[0]
        x_adv = x_adv.detach() + step_size * torch.sign(grad.detach())
        x_adv = torch.min(torch.max(x_adv, x_natural - epsilon), x_natural + epsilon)
        x_adv = torch.clamp(x_adv, 0.0, 1.0)

    model.train()
    x_adv = x_adv.detach()

    optimizer.zero_grad()
    logits_natural = model(x_natural)
    logits_adv = model(x_adv)

    loss_natural = F.cross_entropy(logits_natural, y)
    # batchmean is the numerically-recommended reduction for KLDivLoss --
    # avoids the manual (1/batch_size) scaling of the old "sum" version
    loss_robust = F.kl_div(
        F.log_softmax(logits_adv, dim=1),
        F.softmax(logits_natural, dim=1),
        reduction="batchmean",
    )
    loss = loss_natural + beta * loss_robust
    return loss, logits_natural


for epoch in range(start_epoch, EPOCHS):
    model_trades.train()
    running_loss, correct, total, skipped = 0.0, 0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        loss, logits_natural = trades_loss(model_trades, inputs, labels, optimizer)

        if torch.isnan(loss) or torch.isinf(loss):
            skipped += 1
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_trades.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += loss.item()
        _, predicted = logits_natural.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()
    epoch_loss = running_loss / max(1, (len(train_loader) - skipped))
    epoch_acc = 100. * correct / max(1, total)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | TRADES Loss: {epoch_loss:.4f} | Natural Acc: {epoch_acc:.2f}%"
          + (f" | Skipped {skipped} NaN batches" if skipped else ""))

    if skipped > len(train_loader) * 0.5:
        print(f"[!] Over half the batches were NaN this epoch — stopping. Lower BETA or LEARNING_RATE and restart from scratch.")
        break

    save_checkpoint(model_trades, optimizer, epoch, "TRADES_ResNet18")

torch.save(model_trades.state_dict(), "/kaggle/working/resnet18_trades_cifar10.pt")
print("[*] TRADES training complete! Final model saved.")

In [11]:
import torch
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from attacks.whitebox import fgsm_attack, pgd_attack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

_, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model_trades = get_cifar10_resnet18(num_classes=10)
model_trades.load_state_dict(torch.load("/kaggle/working/resnet18_trades_cifar10.pt"))
model_trades.to(DEVICE)
model_trades.eval()

correct_clean, correct_fgsm, correct_pgd, total = 0, 0, 0, 0
print("[*] Evaluating TRADES model against attacks...")

for inputs, labels in test_loader:
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    total += labels.size(0)

    with torch.no_grad():
        correct_clean += model_trades(inputs).argmax(1).eq(labels).sum().item()

    inputs_fgsm = fgsm_attack(model_trades, inputs, labels, epsilon=8/255)
    with torch.no_grad():
        correct_fgsm += model_trades(inputs_fgsm).argmax(1).eq(labels).sum().item()

    inputs_pgd = pgd_attack(model_trades, inputs, labels, epsilon=8/255, alpha=2/255, iters=10)
    with torch.no_grad():
        correct_pgd += model_trades(inputs_pgd).argmax(1).eq(labels).sum().item()

trades_clean_acc = 100. * correct_clean / total
trades_fgsm_acc = 100. * correct_fgsm / total
trades_pgd_acc = 100. * correct_pgd / total

print("=" * 40)
print(f"TRADES Model Clean Accuracy: {trades_clean_acc:.2f}%")
print(f"TRADES Model FGSM Accuracy:  {trades_fgsm_acc:.2f}%")
print(f"TRADES Model PGD Accuracy:   {trades_pgd_acc:.2f}%")
print("=" * 40)

[*] Evaluating TRADES model against attacks...
TRADES Model Clean Accuracy: 81.88%
TRADES Model FGSM Accuracy:  57.18%
TRADES Model PGD Accuracy:   53.03%


In [ ]:
!pip install -q statsmodels

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 30
SIGMA = 0.25 # Industry standard noise level for CIFAR-10

print(f"[*] Training Gaussian-Augmented Model (sigma={SIGMA})...")
train_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model_gaussian = get_cifar10_resnet18(num_classes=10).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_gaussian.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    model_gaussian.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # ADD GAUSSIAN NOISE (The Cohen et al. requirement)
        noise = torch.randn_like(inputs) * SIGMA
        noisy_inputs = inputs + noise
        
        optimizer.zero_grad()
        outputs = model_gaussian(noisy_inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    scheduler.step()
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Noise Loss: {running_loss/len(train_loader):.4f} | Acc: {100.*correct/total:.2f}%")

torch.save(model_gaussian.state_dict(), "/kaggle/working/resnet18_gaussian_cifar10.pt")
print("[*] Gaussian model saved!")

# --- DEFINE THE SMOOTHING CLASS ---
class Smooth:
    ABSTAIN = -1
    def __init__(self, base_classifier: torch.nn.Module, num_classes: int, sigma: float):
        self.base_classifier = base_classifier
        self.num_classes = num_classes
        self.sigma = sigma

    def certify(self, x: torch.Tensor, n0: int, n: int, alpha: float, batch_size: int) -> tuple[int, float]:
        self.base_classifier.eval()
        counts0 = self._sample_noise(x, n0, batch_size)
        c_hat_A = counts0.argmax().item()

        counts = self._sample_noise(x, n, batch_size)
        nA = counts[c_hat_A].item()

        pA_lower = proportion_confint(nA, n, alpha=2 * alpha, method="beta")[0]

        if pA_lower > 0.5:
            radius = self.sigma * norm.ppf(pA_lower)
            return c_hat_A, radius
        else:
            return self.ABSTAIN, 0.0

    def _sample_noise(self, x: torch.Tensor, num_samples: int, batch_size: int) -> np.ndarray:
        device = next(self.base_classifier.parameters()).device
        x = x.to(device)
        counts = np.zeros(self.num_classes, dtype=int)
        
        with torch.no_grad():
            for _ in range(int(np.ceil(num_samples / batch_size))):
                this_batch_size = min(batch_size, num_samples)
                num_samples -= this_batch_size
                batch = x.repeat((this_batch_size, 1, 1, 1))
                noise = torch.randn_like(batch) * self.sigma
                noisy_batch = batch + noise
                
                outputs = self.base_classifier(noisy_batch)
                predictions = outputs.argmax(dim=1).cpu().numpy()
                for pred in predictions:
                    counts[pred] += 1
        return counts

In [ ]:
import pandas as pd

# 1. Certification Hyperparameters
N0 = 100       # Samples to guess the class
N = 10000      # Samples to estimate the lower bound (reduced from 100k for speed)
ALPHA = 0.001  # 99.9% confidence
SUBSET_SIZE = 250 # Number of images to certify

print(f"[*] Certifying {SUBSET_SIZE} images with N={N}, alpha={ALPHA}, sigma={SIGMA}")

# 2. Initialize Smoothed Classifier
model_gaussian.eval()
smoothed_classifier = Smooth(model_gaussian, num_classes=10, sigma=SIGMA)
_, test_loader = get_dataloaders(batch_size=1) 

results = []

# 3. Run Certification Loop
for i, (x, label) in enumerate(test_loader):
    if i >= SUBSET_SIZE: 
        break
        
    pred, radius = smoothed_classifier.certify(x[0], N0, N, ALPHA, batch_size=256)
    correct = int(pred == label.item())
    
    results.append({
        "image_idx": i,
        "label": label.item(),
        "prediction": pred,
        "correct": correct,
        "radius": radius
    })
    
    if (i + 1) % 10 == 0:
        print(f"Certified {i + 1}/{SUBSET_SIZE} | Last Radius: {radius:.4f} | Correct: {bool(correct)}")

# 4. Export Data
df = pd.DataFrame(results)
df.to_csv("/kaggle/working/certification_radii.csv", index=False)
print("\n[*] SUCCESS: certification_radii.csv saved! Phase 2 is complete.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the certification data
df = pd.read_csv("/kaggle/working/certification_radii.csv")

# 2. Define the radii thresholds to plot (from 0 up to the maximum radius found)
max_radius = df['radius'].max()
radii_thresholds = np.linspace(0, max_radius, 100)
certified_accuracies = []

# 3. Calculate certified accuracy at each threshold
total_images = len(df)
for r in radii_thresholds:
    # A prediction is certified at radius r ONLY IF it was correct AND its certified radius >= r
    robust_count = ((df['correct'] == True) & (df['radius'] >= r)).sum()
    certified_accuracies.append(robust_count / total_images)

# 4. Plot the curve
plt.figure(figsize=(10, 6))
plt.plot(radii_thresholds, certified_accuracies, linewidth=2, color='blue', label=r'Smoothed Classifier ($\sigma=0.25$)')

plt.fill_between(radii_thresholds, certified_accuracies, alpha=0.1, color='blue')
plt.xlabel("Adversarial $L_2$ Radius ($R$)", fontsize=12)
plt.ylabel("Certified Accuracy", fontsize=12)
plt.title("Certified Robustness via Randomized Smoothing", fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim(0, 1.05)
plt.xlim(0, max_radius)
plt.legend(loc='upper right', fontsize=12)

# Save the plot for your report/dashboard
plt.savefig("/kaggle/working/certified_accuracy_curve.png", dpi=300, bbox_inches='tight')
plt.show()

print("[*] Plot saved as certified_accuracy_curve.png! Phase 2 is 100% complete.")

In [ ]:
# ============================================================
# CELL D — Average Certified Radius (ACR)
# Insert right after the certified_accuracy_curve.png cell
# ============================================================
import pandas as pd

SIGMA = 0.25  # must match the sigma used during certification

df = pd.read_csv("/kaggle/working/certification_radii.csv")

# ACR treats an incorrect (or abstained) prediction as radius 0 --
# this is what makes ACR comparable across models/sigmas, since it
# penalizes wrong predictions instead of ignoring them
df["acr_radius"] = df["radius"].where(df["correct"] == True, 0.0)

acr = df["acr_radius"].mean()
n_images = len(df)
n_correct = int(df["correct"].sum())

print("=" * 45)
print(f"Average Certified Radius (ACR)")
print(f"sigma = {SIGMA}, N images = {n_images}")
print("-" * 45)
print(f"Correctly classified & certified: {n_correct}/{n_images} ({100*n_correct/n_images:.1f}%)")
print(f"ACR: {acr:.4f}")
print("=" * 45)

pd.DataFrame([{"sigma": SIGMA, "n_images": n_images, "n_correct": n_correct, "acr": acr}]) \
  .to_csv("/kaggle/working/acr_summary.csv", index=False)
print("[*] Saved acr_summary.csv")

In [12]:
# ============================================================
# CELL B (insert directly after the certified-accuracy plot cell)
# Eval-only: load the existing Gaussian checkpoint, don't retrain.
# ============================================================
import torch
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGMA = 0.25

model_gaussian_eval = get_cifar10_resnet18(num_classes=10)
model_gaussian_eval.load_state_dict(torch.load("/kaggle/working/resnet18_gaussian_cifar10.pt"))
model_gaussian_eval.to(DEVICE)
model_gaussian_eval.eval()

_, test_loader = get_dataloaders(batch_size=128)

correct_clean, correct_noisy, total = 0, 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        total += labels.size(0)
        correct_clean += model_gaussian_eval(inputs).argmax(1).eq(labels).sum().item()
        noisy_inputs = inputs + torch.randn_like(inputs) * SIGMA
        correct_noisy += model_gaussian_eval(noisy_inputs).argmax(1).eq(labels).sum().item()

gaussian_clean_acc = 100. * correct_clean / total
gaussian_noisy_acc = 100. * correct_noisy / total

print(f"Gaussian Model Clean Accuracy (sigma=0):    {gaussian_clean_acc:.2f}%")
print(f"Gaussian Model Accuracy (sigma={SIGMA}):       {gaussian_noisy_acc:.2f}%")

Gaussian Model Clean Accuracy (sigma=0):    73.02%
Gaussian Model Accuracy (sigma=0.25):       79.19%


In [8]:
# ============================================================
# CELL A (replaces old Cells 13 & 14)
# Carlini & Wagner L2 Attack with binary search over c
# ============================================================
import torch
import torch.optim as optim
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders

def cw_l2_attack(model, images, labels, targeted=False, kappa=0,
                  max_iterations=200, lr=0.01,
                  binary_search_steps=9, initial_const=1e-2,
                  c_upper_bound=1e10, c_lower_bound=0.0):
    """
    Carlini & Wagner L2 Attack, with binary search over c per image.
    A fixed c either barely perturbs the image (too small) or ignores
    the L2 term entirely (too large), depending on the model -- that's
    why the old fixed c=1e-3 version only fooled 2/16 images. Binary
    search finds, per image, the smallest c that still succeeds.
    """
    device = images.device
    images = images.clone().detach().to(device)
    labels = labels.clone().detach().to(device)
    batch_size = images.size(0)

    def atanh(x):
        x = torch.clamp(x, -1 + 1e-6, 1 - 1e-6)
        return 0.5 * torch.log((1 + x) / (1 - x))

    x_atanh = atanh((images * 2 - 1) * 0.999999)

    lower_bound = torch.full((batch_size,), c_lower_bound, device=device)
    upper_bound = torch.full((batch_size,), c_upper_bound, device=device)
    const = torch.full((batch_size,), initial_const, device=device)

    best_l2 = torch.full((batch_size,), 1e10, device=device)
    best_adv = images.clone()

    for step in range(binary_search_steps):
        modifier = torch.zeros_like(images, requires_grad=True, device=device)
        optimizer = optim.Adam([modifier], lr=lr)
        step_success = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for it in range(max_iterations):
            optimizer.zero_grad()
            adv_images = 0.5 * (torch.tanh(modifier + x_atanh) + 1)
            l2_dist = torch.sum((adv_images - images) ** 2, dim=[1, 2, 3])

            outputs = model(adv_images)
            real_logits = outputs.gather(1, labels.unsqueeze(1)).squeeze(1)
            outputs_clone = outputs.clone()
            outputs_clone.scatter_(1, labels.unsqueeze(1), -float('inf'))
            other_logits = outputs_clone.max(1)[0]

            if targeted:
                f_loss = torch.clamp(other_logits - real_logits + kappa, min=0)
            else:
                f_loss = torch.clamp(real_logits - other_logits + kappa, min=0)

            loss = torch.sum(l2_dist + const * f_loss)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                preds = outputs.argmax(1)
                success = (preds != labels) if not targeted else (preds == labels)
                step_success = step_success | success
                improved = success & (l2_dist < best_l2)
                best_l2 = torch.where(improved, l2_dist, best_l2)
                if improved.any():
                    best_adv[improved] = adv_images[improved].detach()

        for i in range(batch_size):
            if step_success[i]:
                upper_bound[i] = min(upper_bound[i].item(), const[i].item())
                const[i] = (lower_bound[i] + upper_bound[i]) / 2
            else:
                lower_bound[i] = max(lower_bound[i].item(), const[i].item())
                if upper_bound[i] < c_upper_bound:
                    const[i] = (lower_bound[i] + upper_bound[i]) / 2
                else:
                    const[i] *= 10

    return best_adv


# --- Re-run the demo eval on the baseline model with the corrected attack ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_baseline = get_cifar10_resnet18(num_classes=10)
model_baseline.load_state_dict(torch.load("/kaggle/working/resnet18_standard_cifar10.pt"))
model_baseline.to(DEVICE)
model_baseline.eval()

_, test_loader = get_dataloaders(batch_size=16)
inputs, labels = next(iter(test_loader))
inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

print("[*] Running C&W (binary search, 9 steps x 200 iters)...")

with torch.no_grad():
    clean_acc = model_baseline(inputs).argmax(1).eq(labels).float().mean().item()

adv_inputs = cw_l2_attack(model_baseline, inputs, labels, max_iterations=200, binary_search_steps=9)

with torch.no_grad():
    adv_acc = model_baseline(adv_inputs).argmax(1).eq(labels).float().mean().item()
    avg_l2 = torch.sqrt(torch.sum((adv_inputs - inputs) ** 2, dim=[1, 2, 3])).mean().item()

print("-" * 40)
print(f"Clean Batch Accuracy: {clean_acc * 100:.2f}%")
print(f"C&W Batch Accuracy:   {adv_acc * 100:.2f}%")
print(f"Average L2 Noise:     {avg_l2:.4f}")
print("-" * 40)

[*] Running C&W (binary search, 9 steps x 200 iters)...
----------------------------------------
Clean Batch Accuracy: 100.00%
C&W Batch Accuracy:   0.00%
Average L2 Noise:     0.1338
----------------------------------------


In [9]:
# ============================================================
# CELL F — C&W L2 attack against PGD-AT and TRADES
# Fills in the previously-blank cw_l2_acc / cw_l2_avg_l2_norm
# cells for these two models. Requires cw_l2_attack from Cell A
# to already be defined in this session.
# ============================================================
import torch
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, test_loader = get_dataloaders(batch_size=16)
inputs, labels = next(iter(test_loader))
inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)


def evaluate_cw(model_path, model_name):
    model = get_cifar10_resnet18(num_classes=10)
    model.load_state_dict(torch.load(model_path))
    model.to(DEVICE)
    model.eval()

    with torch.no_grad():
        clean_acc = model(inputs).argmax(1).eq(labels).float().mean().item()

    adv_inputs = cw_l2_attack(model, inputs, labels, max_iterations=200, binary_search_steps=9)

    with torch.no_grad():
        adv_acc = model(adv_inputs).argmax(1).eq(labels).float().mean().item()
        avg_l2 = torch.sqrt(torch.sum((adv_inputs - inputs) ** 2, dim=[1, 2, 3])).mean().item()

    print(f"[{model_name}] Clean: {clean_acc*100:.2f}% | C&W: {adv_acc*100:.2f}% | Avg L2: {avg_l2:.4f}")
    return adv_acc * 100, avg_l2


print("[*] Running C&W against PGD-AT and TRADES (same 16-image batch used for the baseline)...")
pgdat_cw_acc, pgdat_cw_l2 = evaluate_cw("/kaggle/working/resnet18_robust_pgd_cifar10.pt", "PGD-AT")
trades_cw_acc, trades_cw_l2 = evaluate_cw("/kaggle/working/resnet18_trades_cifar10.pt", "TRADES")

[*] Running C&W against PGD-AT and TRADES (same 16-image batch used for the baseline)...
[PGD-AT] Clean: 87.50% | C&W: 0.00% | Avg L2: 0.7746
[TRADES] Clean: 87.50% | C&W: 0.00% | Avg L2: 0.8197


In [ ]:
import torch
import pandas as pd
import numpy as np
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint

# 1. Setup & Load Smoothed Model
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGMA = 0.25

model_gaussian = get_cifar10_resnet18(num_classes=10)
model_gaussian.load_state_dict(torch.load("/kaggle/working/resnet18_gaussian_cifar10.pt"))
model_gaussian.to(DEVICE)
model_gaussian.eval()

# Re-define Smooth class briefly for standalone prediction
class SmoothPredictor:
    def __init__(self, base_classifier, num_classes, sigma):
        self.base_classifier = base_classifier
        self.num_classes = num_classes
        self.sigma = sigma
        
    def predict(self, x: torch.Tensor, n: int, batch_size: int):
        self.base_classifier.eval()
        counts = np.zeros(self.num_classes, dtype=int)
        x = x.to(DEVICE)
        
        with torch.no_grad():
            for _ in range(int(np.ceil(n / batch_size))):
                this_batch_size = min(batch_size, n)
                n -= this_batch_size
                batch = x.repeat((this_batch_size, 1, 1, 1))
                noise = torch.randn_like(batch) * self.sigma
                outputs = self.base_classifier(batch + noise)
                preds = outputs.argmax(dim=1).cpu().numpy()
                for p in preds: counts[p] += 1
        return counts.argmax().item()

smoothed_model = SmoothPredictor(model_gaussian, 10, SIGMA)

# 2. Pick a strongly certified image from Phase 2
df = pd.read_csv("/kaggle/working/certification_radii.csv")
# Find an image that was correctly classified and has a large certified radius
robust_images = df[(df['correct'] == True) & (df['radius'] > 0.4)]
target_idx = robust_images.iloc[0]['image_idx']
certified_R = robust_images.iloc[0]['radius']
true_label = robust_images.iloc[0]['label']

print(f"[*] SANITY TEST: Image Index {target_idx}")
print(f"[*] Certified Safe Radius (R): {certified_R:.4f}")
print(f"[*] True Label: {true_label}")
print("-" * 40)

# 3. Retrieve the actual image
_, test_loader = get_dataloaders(batch_size=1)
for i, (x, y) in enumerate(test_loader):
    if i == target_idx:
        clean_image = x.to(DEVICE)
        break

# 4. Generate C&W Attack (High 'c' to ensure it tries hard to attack)
print("[*] Generating C&W Attack to find an adversarial direction...")
# OLD: adv_image = cw_l2_attack(model_gaussian, clean_image, y.to(DEVICE), c=0.5, max_iters=500)
adv_image = cw_l2_attack(model_gaussian, clean_image, y.to(DEVICE), max_iterations=200)

# 5. Scale the perturbation to be STRICTLY LESS than the certified radius R
perturbation = adv_image - clean_image
current_l2 = torch.sqrt(torch.sum(perturbation ** 2)).item()

# We scale it to 95% of R so it is definitively inside the certified safe zone
scaling_factor = (certified_R * 0.95) / (current_l2 + 1e-8) 
bounded_perturbation = perturbation * scaling_factor
bounded_adv_image = torch.clamp(clean_image + bounded_perturbation, 0, 1)

final_l2 = torch.sqrt(torch.sum((bounded_adv_image - clean_image) ** 2)).item()
print(f"[*] Scaled Attack L2 Norm: {final_l2:.4f} (Strictly < {certified_R:.4f})")

# 6. The Moment of Truth: Can it fool the smoothed classifier?
print("[*] Passing bounded attack to the Smoothed Classifier (N=1000)...")
robust_pred = smoothed_model.predict(bounded_adv_image[0], n=1000, batch_size=256)

print("-" * 40)
print(f"[*] Smoothed Model Prediction: {robust_pred}")
if robust_pred == true_label:
    print("[*] RESULT: SUCCESS! The certified boundary held. Prediction did not flip.")
else:
    print("[*] RESULT: FAILURE. The mathematical bound was broken.")
print("-" * 40)

In [ ]:
# ============================================================
# CELL E — Scaled sanity test across all certified images
# Insert right after the existing single-image sanity-test cell
# ============================================================
import torch
import pandas as pd
import numpy as np
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SIGMA = 0.25
SAMPLE_SIZE = 50  # out of 250 certified images -- see cost note below

model_gaussian_full = get_cifar10_resnet18(num_classes=10)
model_gaussian_full.load_state_dict(torch.load("/kaggle/working/resnet18_gaussian_cifar10.pt"))
model_gaussian_full.to(DEVICE)
model_gaussian_full.eval()

class SmoothPredictor:
    def __init__(self, base_classifier, num_classes, sigma):
        self.base_classifier = base_classifier
        self.num_classes = num_classes
        self.sigma = sigma

    def predict(self, x, n, batch_size):
        self.base_classifier.eval()
        counts = np.zeros(self.num_classes, dtype=int)
        x = x.to(DEVICE)
        with torch.no_grad():
            remaining = n
            while remaining > 0:
                this_batch = min(batch_size, remaining)
                remaining -= this_batch
                batch = x.repeat((this_batch, 1, 1, 1))
                noise = torch.randn_like(batch) * self.sigma
                preds = self.base_classifier(batch + noise).argmax(dim=1).cpu().numpy()
                for p in preds:
                    counts[p] += 1
        return counts.argmax().item()

smoothed_model_full = SmoothPredictor(model_gaussian_full, 10, SIGMA)

df = pd.read_csv("/kaggle/working/certification_radii.csv")
certified_images = df[(df["correct"] == True) & (df["radius"] > 0)] \
    .sample(n=min(SAMPLE_SIZE, (df["correct"] == True).sum()), random_state=42) \
    .reset_index(drop=True)

print(f"[*] Running scaled sanity test on {len(certified_images)} certified images...")

_, test_loader = get_dataloaders(batch_size=1)
target_indices = set(certified_images["image_idx"].astype(int))
image_lookup = {}
for i, (x, y) in enumerate(test_loader):
    if i in target_indices:
        image_lookup[i] = (x, y)
    if len(image_lookup) == len(target_indices):
        break

held, results = 0, []
for _, row in certified_images.iterrows():
    idx, certified_R, true_label = int(row["image_idx"]), row["radius"], int(row["label"])
    x, y = image_lookup[idx]
    clean_image, y = x.to(DEVICE), y.to(DEVICE)

    adv_image = cw_l2_attack(model_gaussian_full, clean_image, y, max_iterations=100, binary_search_steps=5)

    perturbation = adv_image - clean_image
    current_l2 = torch.sqrt(torch.sum(perturbation ** 2)).item()
    scaling_factor = (certified_R * 0.95) / (current_l2 + 1e-8)
    bounded_adv_image = torch.clamp(clean_image + perturbation * scaling_factor, 0, 1)

    robust_pred = smoothed_model_full.predict(bounded_adv_image[0], n=1000, batch_size=256)
    success = int(robust_pred == true_label)
    held += success
    results.append({"image_idx": idx, "radius": certified_R, "bound_held": bool(success)})

print("=" * 45)
print(f"Certified bound held: {held}/{len(certified_images)} ({100*held/len(certified_images):.2f}%)")
print("=" * 45)

pd.DataFrame(results).to_csv("/kaggle/working/sanity_test_scaled.csv", index=False)
print("[*] Saved sanity_test_scaled.csv")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v2
import pandas as pd
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from attacks.whitebox import pgd_attack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS_SUBSTITUTE = 10  # Just enough to learn a decision boundary

print("[*] Phase 4: Building the Transferability Matrix")
train_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)

# 1. Train the Substitute Model (MobileNetV2)
print("\n[*] Training Substitute Model (MobileNetV2)...")
model_substitute = mobilenet_v2(num_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_substitute.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_SUBSTITUTE)

for epoch in range(EPOCHS_SUBSTITUTE):
    model_substitute.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_substitute(inputs), labels)
        loss.backward()
        optimizer.step()
    scheduler.step()
    print(f"    Epoch {epoch+1}/{EPOCHS_SUBSTITUTE} complete.")

model_substitute.eval()

# 2. Load Your Baseline Target (ResNet18)
print("\n[*] Loading Target Model (Baseline ResNet18)...")
model_target = get_cifar10_resnet18(num_classes=10)
model_target.load_state_dict(torch.load("/kaggle/working/resnet18_standard_cifar10.pt"))
model_target.to(DEVICE)
model_target.eval()

# 3. Cross-Attack Evaluation Logic
def evaluate_transfer(source_model, target_model, dataloader, attack_name="PGD"):
    correct_clean, correct_adv, total = 0, 0, 0
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        total += labels.size(0)
        
        # Target's Clean Accuracy
        with torch.no_grad():
            _, clean_preds = target_model(inputs).max(1)
            correct_clean += clean_preds.eq(labels).sum().item()
            
        # Generate Attack on SOURCE
        adv_inputs = pgd_attack(source_model, inputs, labels, epsilon=8/255, alpha=2/255, iters=10)
        
        # Test Attack on TARGET
        with torch.no_grad():
            _, adv_preds = target_model(adv_inputs).max(1)
            correct_adv += adv_preds.eq(labels).sum().item()
            
        # Break early for speed (test on a subset of 5 batches)
        if total >= 5 * BATCH_SIZE:
            break
            
    return (correct_clean / total) * 100, (correct_adv / total) * 100

# 4. Generate the Matrix
print("\n[*] Running Cross-Attacks...")

# A. White-Box: ResNet attacks ResNet
_, resnet_to_resnet = evaluate_transfer(model_target, model_target, test_loader)
# B. Black-Box: MobileNet attacks ResNet
_, mobilenet_to_resnet = evaluate_transfer(model_substitute, model_target, test_loader)
# C. Black-Box: ResNet attacks MobileNet
_, resnet_to_mobilenet = evaluate_transfer(model_target, model_substitute, test_loader)
# D. White-Box: MobileNet attacks MobileNet
clean_mobile, mobilenet_to_mobilenet = evaluate_transfer(model_substitute, model_substitute, test_loader)

# 5. Save and Display
matrix = pd.DataFrame({
    "Target: ResNet18": [resnet_to_resnet, mobilenet_to_resnet],
    "Target: MobileNetV2": [resnet_to_mobilenet, mobilenet_to_mobilenet]
}, index=["Source: ResNet18", "Source: MobileNetV2"])

matrix.to_csv("/kaggle/working/transferability_matrix.csv")

print("\n" + "="*45)
print("       TRANSFERABILITY MATRIX (Accuracy %)")
print("="*45)
print(matrix.round(2))
print("="*45)
print("[*] Saved to transferability_matrix.csv!")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load Both Models
model_std = get_cifar10_resnet18(num_classes=10).to(DEVICE)
model_std.load_state_dict(torch.load("/kaggle/working/resnet18_standard_cifar10.pt"))
model_std.eval()

model_rob = get_cifar10_resnet18(num_classes=10).to(DEVICE)
model_rob.load_state_dict(torch.load("/kaggle/working/resnet18_robust_pgd_cifar10.pt"))
model_rob.eval()

# 2. Get a single image and label
_, test_loader = get_dataloaders(batch_size=1)
images, labels = next(iter(test_loader))
img, label = images.to(DEVICE), labels.to(DEVICE)
criterion = nn.CrossEntropyLoss()

print("[*] Generating Loss Landscapes... (This takes a moment to map the grid)")

# 3. Find the Adversarial Direction (X-axis)
img.requires_grad = True
loss = criterion(model_std(img), label)
loss.backward()
d1 = img.grad.detach().sign() # Fast Gradient Sign direction
img.requires_grad = False

# 4. Generate an Orthogonal Random Direction (Y-axis) via Gram-Schmidt
d2 = torch.randn_like(d1)
# Projection of d2 onto d1
proj = (torch.sum(d2 * d1) / torch.sum(d1 * d1)) * d1
d2 = d2 - proj # Make it orthogonal
d2 = d2 / torch.norm(d2) * torch.norm(d1) # Scale it to have the same magnitude as d1

# 5. Define the Grid (Scanning from -epsilon to +epsilon)
EPSILON = 8 / 255
GRID_SIZE = 20 # 20x20 grid (400 forward passes per model)
alphas = np.linspace(-EPSILON * 2, EPSILON * 2, GRID_SIZE)
betas = np.linspace(-EPSILON * 2, EPSILON * 2, GRID_SIZE)

loss_surface_std = np.zeros((GRID_SIZE, GRID_SIZE))
loss_surface_rob = np.zeros((GRID_SIZE, GRID_SIZE))

# 6. Map the Terrain
with torch.no_grad():
    for i, alpha in enumerate(alphas):
        for j, beta in enumerate(betas):
            # Perturb the image: x' = x + alpha*d1 + beta*d2
            perturbed_img = img + (alpha * d1) + (beta * d2)
            perturbed_img = torch.clamp(perturbed_img, 0, 1)
            
            # Calculate loss for Standard Model
            out_std = model_std(perturbed_img)
            loss_surface_std[i, j] = criterion(out_std, label).item()
            
            # Calculate loss for Robust Model
            out_rob = model_rob(perturbed_img)
            loss_surface_rob[i, j] = criterion(out_rob, label).item()

# 7. Plotting the Contours
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Standard Model Plot
C1 = ax1.contourf(alphas, betas, loss_surface_std, levels=20, cmap='Reds')
fig.colorbar(C1, ax=ax1)
ax1.set_title("Standard Model (Steep Cliff)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Adversarial Direction", fontsize=12)
ax1.set_ylabel("Random Orthogonal Direction", fontsize=12)

# Robust Model Plot
C2 = ax2.contourf(alphas, betas, loss_surface_rob, levels=20, cmap='Blues')
fig.colorbar(C2, ax=ax2)
ax2.set_title("Robust Model (Flat Plateau)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Adversarial Direction", fontsize=12)
ax2.set_ylabel("Random Orthogonal Direction", fontsize=12)

plt.suptitle("2D Loss Landscape Comparison", fontsize=18, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig("/kaggle/working/loss_landscape_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print("[*] Saved to loss_landscape_comparison.png!")

In [13]:
# cell c
import pandas as pd

empirical_benchmarks = pd.DataFrame([
    {
        "model": "Baseline (Standard)",
        "clean_acc": 94.38, "fgsm_acc": 31.72, "pgd10_acc": 0.02,
        "cw_l2_acc": adv_acc * 100, "cw_l2_avg_l2_norm": avg_l2,
    },
    {
        "model": "PGD Adversarial Training",
        "clean_acc": 82.74, "fgsm_acc": 55.42, "pgd10_acc": 49.51,
        "cw_l2_acc": pgdat_cw_acc, "cw_l2_avg_l2_norm": pgdat_cw_l2,
    },
    {
        "model": "TRADES",
        "clean_acc": trades_clean_acc, "fgsm_acc": trades_fgsm_acc, "pgd10_acc": trades_pgd_acc,
        "cw_l2_acc": trades_cw_acc, "cw_l2_avg_l2_norm": trades_cw_l2,
    },
    {
        "model": "Gaussian Augmented (sigma=0.25)",
        "clean_acc": gaussian_clean_acc, "fgsm_acc": None, "pgd10_acc": None,
        "cw_l2_acc": None, "cw_l2_avg_l2_norm": None,
    },
])

empirical_benchmarks.to_csv("/kaggle/working/empirical_benchmarks.csv", index=False)
print("[*] Saved empirical_benchmarks.csv")
print(empirical_benchmarks)

[*] Saved empirical_benchmarks.csv
                             model  clean_acc  fgsm_acc  pgd10_acc  cw_l2_acc  \
0              Baseline (Standard)      94.38     31.72       0.02        0.0   
1         PGD Adversarial Training      82.74     55.42      49.51        0.0   
2                           TRADES      81.88     57.18      53.03        0.0   
3  Gaussian Augmented (sigma=0.25)      73.02       NaN        NaN        NaN   

   cw_l2_avg_l2_norm  
0           0.133840  
1           0.774579  
2           0.819690  
3                NaN  


In [7]:
"""
Run this ONCE in your Kaggle session to export a small, self-contained
subset of CIFAR-10 test images for the dashboard.

Why this exists: the deployed dashboard (Streamlit Community Cloud / HF
Spaces) has no access to /kaggle/input, so it needs its own bundled copy
of a few demo images rather than pulling from the full test set live.

Prerequisites: repo already cloned and on sys.path (same as your training
notebook), so `utils.dataset` is importable.

Output: demo_images.pt -- copy this into dashboard/assets/ in your repo.
"""

import torch
from utils.dataset import get_dataloaders

N_IMAGES = 40  # keep this small -- it ships inside the dashboard repo

_, test_loader = get_dataloaders(batch_size=1)

images, labels = [], []
for i, (x, y) in enumerate(test_loader):
    if i >= N_IMAGES:
        break
    images.append(x.squeeze(0))
    labels.append(y.item())

bundle = {
    "images": torch.stack(images),  # [N, 3, 32, 32], values in [0, 1]
    "labels": torch.tensor(labels),
}
torch.save(bundle, "/kaggle/working/demo_images.pt")
print(f"[*] Saved demo_images.pt with {N_IMAGES} images")
print("[*] Copy this file into dashboard/assets/demo_images.pt")


[*] Saved demo_images.pt with 40 images
[*] Copy this file into dashboard/assets/demo_images.pt
